In [6]:
# GPU 살펴보기
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available()else "cpu")


True
Tesla T4


In [7]:
# GPU로 옮기기
inputs= torch.randn(1000, 1000)
inputs= inputs.to(device)
print(inputs.device)

cuda:0


In [8]:
# 데이터 이동 시간 측정
import torch
import time

device = "cuda"
x = torch.randn(10000,10000)

torch.cuda.synchronize()
start = time.time()

x_gpu = x.to(device)

torch.cuda.synchronize()
print("CPU → GPU:", time.time() - start)

torch.cuda.synchronize()
start = time.time()
y_gpu = x_gpu * 2

torch.cuda.synchronize()
print("GPU computation:", time.time() - start)

# GPU → CPU
torch.cuda.synchronize()
start = time.time()
y_cpu = y_gpu.cpu()
torch.cuda.synchronize()
print("GPU → CPU:",time.time() - start)

# GPU는 계산은 엄청 빠른데, 데이터를 GPU로 옮기고 다시 가져오는 게 훨씬 느림 그러니까 옮기는 횟수를 최대한 줄이자

CPU → GPU: 0.08540534973144531
GPU computation: 0.031421661376953125
GPU → CPU: 0.2725248336791992


In [9]:
# 행렬 연산
import torch
import time

N = 5000

A = torch.randn(N, N)
B = torch.randn(N, N)
start = time.time()
C = torch.matmul(A, B)
print( "CPU time:", time.time() - start )

A = A.cuda()
B = B.cuda()

torch.cuda.synchronize()
start = time.time()

C = torch.matmul(A, B)

torch.cuda.synchronize() # synchronize로 GPU 끝날 때까지 기다리도록 하기
print( "GPU time:", time.time() - start )
# 당연히 GPU의 시간이 더 짧은 것을 알 수 있음!

CPU time: 1.9051096439361572
GPU time: 0.08797574043273926


In [10]:
# vRAM 살펴보기
print( torch.cuda.memory_allocated() / 1024**2, "MB")

# 늘려보기
x = torch.randn( 100000, 100000, device="cuda")
# OutOfMemoryError: CUDA out of memory.

print( torch.cuda.memory_allocated() / 1024**2, "MB")


1207.93994140625 MB


OutOfMemoryError: CUDA out of memory. Tried to allocate 37.25 GiB. GPU 0 has a total capacity of 14.56 GiB of which 13.25 GiB is free. Including non-PyTorch memory, this process has 1.31 GiB memory in use. Of the allocated memory 1.18 GiB is allocated by PyTorch, and 8.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [11]:
# Precision & Memory
x32 = torch.randn( 5000, 5000, device= "cuda", dtype=torch.float32)
memory32 = (x32.element_size() * x32.nelement())
print(memory32 / 1024**2 , "MB")

x16 = torch.randn( 5000, 5000, device= "cuda", dtype=torch.float16)
memory16 = (x16.element_size() * x16.nelement())
print(memory16 / 1024**2 , "MB")

# 원소 개수는 같고, 원소 1개당 바이트만 4 -> 2로 감소
# 메모리도 정확히 절반
# 즉, 정밀도를 낮추면 같은 VRAM에 2배 큰 모델/배치를 올릴 수 있음

95.367431640625 MB
47.6837158203125 MB
